# Use Case 04: Financial Earnings Call Analyzer

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/rajvermatx/careeralign.com/blob/main/genai/usecases/notebooks/04-earnings-call-analyzer.ipynb)

This notebook builds a complete **Earnings Call Analyzer** pipeline that takes a raw earnings call transcript and produces:

1. **Speaker identification** — who said what (CEO, CFO, Analyst)
2. **Topic segmentation** — breaking the call into thematic sections
3. **Financial metric extraction** — pulling out revenue, EPS, margins, guidance
4. **Sentiment analysis** — per-topic tone scoring
5. **Executive summary** — a structured analyst brief with citations

We use a **synthetic earnings call transcript** for a fictional company (NovaTech Solutions) so the notebook runs end-to-end without external data dependencies.

## Setup

In [ ]:
!pip install -q openai pandas matplotlib

In [ ]:
import os
import re
import json
import pandas as pd
import matplotlib.pyplot as plt
from openai import OpenAI
from typing import List, Dict

# Set your API key (use Colab secrets or environment variable)
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

client = OpenAI()
MODEL = "gpt-4o-mini"  # Use gpt-4o for higher accuracy in production

print("Setup complete.")

## 1. Synthetic Earnings Call Transcript

Below is a realistic Q3 FY2025 earnings call transcript for **NovaTech Solutions** (ticker: NVTS), a fictional mid-cap enterprise software company. The transcript includes prepared remarks from the CEO and CFO, followed by analyst Q&A.

In [ ]:
TRANSCRIPT = """
Operator: Good afternoon, everyone, and welcome to the NovaTech Solutions Third Quarter Fiscal Year 2025 Earnings Call. Today's call is being recorded. All participants are in listen-only mode. After the prepared remarks, we will open the call for questions. I would now like to turn the call over to Sarah Chen, Vice President of Investor Relations. Sarah, please go ahead.

Sarah Chen, VP Investor Relations: Thank you, operator. Good afternoon and thank you for joining us today. With me on the call are David Park, our Chief Executive Officer, and Lisa Martinez, our Chief Financial Officer. Before we begin, I want to remind everyone that during this call we will make forward-looking statements that involve risks and uncertainties. Actual results may differ materially from those discussed today. Please refer to our SEC filings for a detailed discussion of risk factors. We will also discuss certain non-GAAP financial measures. Reconciliations to the most comparable GAAP measures are available in our earnings press release. With that, I will turn the call over to David.

David Park, CEO: Thank you, Sarah, and good afternoon everyone. I am pleased to report that NovaTech delivered another strong quarter with results that exceeded our expectations across all key metrics. Let me start with the headlines.

Total revenue for Q3 was $2.87 billion, up 18% year-over-year, beating the high end of our guidance range by approximately $70 million. This marks our sixth consecutive quarter of accelerating revenue growth, and I want to emphasize that this acceleration is broad-based across both our Cloud Platform and Enterprise Software segments.

Our Cloud Platform segment continues to be the primary growth engine. Cloud revenue reached $1.92 billion, growing 27% year-over-year and now representing 67% of total revenue, up from 62% a year ago. We added 340 net new enterprise cloud customers in the quarter, bringing our total enterprise cloud customer count to 4,850. Our net revenue retention rate improved to 128%, up from 124% last quarter, reflecting both strong expansion within existing accounts and very low churn.

In our Enterprise Software segment, revenue was $950 million, up 4% year-over-year. While this segment grows more modestly, it remains highly profitable and provides the installed base from which we drive cloud migration. We converted 85 on-premise customers to our cloud platform during the quarter, each with an average contract value increase of 2.3x upon migration.

I want to highlight three strategic initiatives that are driving our momentum. First, our AI-powered analytics suite, which we launched in Q1, is seeing exceptional adoption. Over 1,200 customers are now actively using our AI features, and customers deploying AI analytics are seeing 40% higher engagement scores compared to our traditional analytics. This is translating directly into larger deal sizes — our average annual contract value for AI-inclusive deals is $385,000, compared to $210,000 for standard platform deals.

Second, our international expansion is gaining traction. International revenue grew 31% year-over-year and now represents 28% of total revenue, up from 24% a year ago. We opened new offices in Tokyo and Munich during the quarter and hired 120 sales and customer success professionals across EMEA and APAC. We are particularly excited about the momentum we are seeing in Japan, where we signed three deals valued at over $5 million each.

Third, our partnership ecosystem continues to expand. We now have over 450 technology and channel partners, up from 380 a year ago. Partner-sourced pipeline grew 45% year-over-year, and partner-influenced revenue represented 35% of new bookings in the quarter.

Before turning it over to Lisa for the financial details, I want to acknowledge the macroeconomic environment. We are seeing some lengthening of sales cycles in certain verticals, particularly in financial services and media. However, our win rates have actually improved from 34% to 38% over the past two quarters, suggesting that when companies do invest, they are increasingly choosing NovaTech. We view this as a share-gain dynamic that positions us well for when spending accelerates.

With that, let me turn it over to Lisa. Lisa?

Lisa Martinez, CFO: Thank you, David, and good afternoon everyone. Let me walk through the financial details.

As David mentioned, total revenue was $2.87 billion, up 18% year-over-year. On a constant currency basis, revenue growth was 19%, as FX headwinds reduced reported growth by approximately 1 percentage point.

Gross margin was 74.2% on a GAAP basis, up 120 basis points year-over-year. Non-GAAP gross margin was 76.8%, up 90 basis points. The improvement was driven by better cloud infrastructure utilization — our compute costs per unit of consumption declined 15% as we benefited from our multi-cloud optimization initiative and renegotiated contracts with our major cloud providers.

Operating expenses totaled $1.52 billion on a GAAP basis. Non-GAAP operating expenses were $1.33 billion, or 46.3% of revenue, down from 48.1% a year ago. We are demonstrating meaningful operating leverage while continuing to invest aggressively in R&D and go-to-market.

R&D spending was $620 million, or 21.6% of revenue, compared to 22.4% a year ago. We added 280 engineers during the quarter, with a focus on AI and machine learning capabilities. Despite the absolute increase in R&D spending, we are getting more efficient — our revenue per R&D dollar improved from $4.10 to $4.63 year-over-year.

Sales and marketing expense was $580 million, or 20.2% of revenue, compared to 21.3% a year ago. Our customer acquisition cost decreased 12% year-over-year while average deal size increased 15%, reflecting improved go-to-market efficiency and the premium pricing power of our AI-inclusive offerings.

Non-GAAP operating income was $876 million, representing a non-GAAP operating margin of 30.5%, up from 27.8% a year ago. This is the fourth consecutive quarter of operating margin expansion, and we remain on track to reach our target of 32% non-GAAP operating margin by the end of fiscal year 2026.

Non-GAAP earnings per share was $2.18, up 24% year-over-year and $0.12 above the high end of our guidance range. GAAP EPS was $1.52.

Turning to the balance sheet and cash flow. We generated $780 million of free cash flow in the quarter, representing a 27.2% free cash flow margin. This is up from $620 million and 25.5% in Q3 of last year. Cash and investments at quarter end totaled $8.4 billion.

Remaining performance obligations, or RPO, were $11.2 billion, up 22% year-over-year. Current RPO, which we expect to recognize as revenue over the next 12 months, was $5.8 billion, up 20% year-over-year. This gives us strong visibility into our near-term revenue trajectory.

Now let me turn to guidance. For Q4 fiscal year 2025, we expect total revenue of $3.00 to $3.05 billion, representing year-over-year growth of 17% to 19%. We expect non-GAAP operating margin of 30.5% to 31.0% and non-GAAP EPS of $2.22 to $2.30.

For the full fiscal year 2025, we are raising our revenue guidance to $10.95 to $11.00 billion, up from our prior guidance of $10.70 to $10.80 billion. We now expect full-year non-GAAP operating margin of approximately 29.5%, up from our prior expectation of 28.5% to 29.0%. Full-year non-GAAP EPS is now expected to be $8.05 to $8.13, up from our prior range of $7.70 to $7.85.

I also want to provide an initial framework for fiscal year 2026. We expect revenue growth in the range of 16% to 18%, with continued operating margin expansion toward our 32% target. We will provide more specific FY26 guidance on our Q4 call.

Finally, we repurchased $400 million of stock during the quarter and our board has authorized an additional $2 billion share repurchase program, reflecting our confidence in the business and our commitment to returning capital to shareholders.

With that, let me turn it back to Sarah to open the Q&A.

Sarah Chen, VP Investor Relations: Thank you, Lisa. Operator, we are ready for questions.

Operator: Thank you. Our first question comes from Michael Torres at Morgan Stanley. Please go ahead.

Michael Torres, Analyst, Morgan Stanley: Great, thanks for taking the question. David, you mentioned lengthening sales cycles in financial services and media. Can you quantify that? Are we talking weeks or months of elongation? And are you seeing any similar dynamics in your other verticals?

David Park, CEO: Great question, Michael. In financial services, we have seen average sales cycles extend from approximately 90 days to about 110 days over the past two quarters. In media, it is a bit more pronounced — roughly 85 days to 120 days. However, I want to be very clear that our overall pipeline has not contracted. The deals are taking longer to close, but they are not falling out of pipeline. Our pipeline coverage ratio is 3.2x, which is actually the highest it has been in two years.

In our other major verticals — technology, healthcare, and manufacturing — we have not seen meaningful elongation. In fact, healthcare has actually shortened slightly as organizations accelerate their digital transformation investments. And our win rates across all verticals have improved, as I mentioned, from 34% to 38%.

Michael Torres, Analyst, Morgan Stanley: Very helpful. And Lisa, a quick follow-up on free cash flow. The 27.2% margin was quite strong. How much of that was timing-related versus structural? Should we think of mid-to-high 20s as the sustainable run rate?

Lisa Martinez, CFO: Good question, Michael. I would say about 2 percentage points of the Q3 free cash flow margin benefited from favorable timing of collections — we had some large enterprise invoices that came in earlier than expected. On a normalized basis, I would point you toward a sustainable range of 25% to 26% free cash flow margin, with gradual improvement as operating margins expand. By FY26, we expect to sustain 26% to 28% free cash flow margins.

Operator: Our next question comes from Jennifer Walsh at Goldman Sachs. Please go ahead.

Jennifer Walsh, Analyst, Goldman Sachs: Thank you. I want to dig into the AI analytics adoption. You said 1,200 customers are using it. What is the conversion rate from trial to paid? And what is the incremental margin profile of the AI features?

David Park, CEO: Jennifer, we are seeing a trial-to-paid conversion rate of approximately 65%, which is significantly above our initial target of 50%. Customers are realizing value from the AI features very quickly — our time-to-value metric for AI analytics is about 14 days, compared to 45 days for our traditional platform. That fast time-to-value is driving the high conversion.

In terms of the financial profile, I will let Lisa speak to the margins.

Lisa Martinez, CFO: Thanks, David. The incremental margin on AI features is actually slightly lower than our blended average today, primarily because of the compute costs associated with running inference at scale. Our AI gross margin is approximately 68% compared to our overall 76.8% non-GAAP gross margin. However, we expect this to improve significantly as we optimize our inference infrastructure and as scale benefits kick in. By the second half of FY26, we expect AI feature margins to converge with our overall cloud margin.

The important point is that AI features are driving significantly larger deal sizes — that $385,000 ACV versus $210,000 for standard deals that David mentioned. So while the percentage margin is lower, the absolute margin dollars per customer are substantially higher.

Jennifer Walsh, Analyst, Goldman Sachs: That is really helpful. One more if I may. On international, the 31% growth is impressive. How should we think about the investment trajectory there? Will international expansion be a headwind to margins in the near term?

Lisa Martinez, CFO: We are investing ahead of the revenue opportunity in international, particularly in Japan and Germany where we see large addressable markets. International operating margins are currently about 15 percentage points below our company average, primarily due to the upfront investment in headcount and facilities. We expect international margins to improve by 3 to 5 percentage points per year as these markets mature. By FY27, we expect international margins to be within 5 percentage points of the company average. So yes, there is a near-term margin dilutive effect, but we believe the long-term returns more than justify the investment.

Operator: Our next question comes from Robert Kim at JPMorgan. Please go ahead.

Robert Kim, Analyst, JPMorgan: Thanks. Lisa, on the guidance raise — you raised full-year revenue guidance by $200 million at the midpoint. How much of the raise is from Q3 outperformance flowing through versus an improved outlook for Q4?

Lisa Martinez, CFO: Good question, Robert. I would characterize the raise as roughly 60% Q3 beat flow-through and 40% improved Q4 outlook. Our Q4 pipeline is very strong — RPO growth of 22% and current RPO growth of 20% give us good visibility. We are also seeing strong end-of-fiscal-year budget flush dynamics from several large enterprise customers.

Robert Kim, Analyst, JPMorgan: And David, on competitive dynamics — you mentioned improving win rates. Are you seeing those gains primarily against the large horizontal platforms or the vertical-specific competitors?

David Park, CEO: Robert, we are gaining share on both fronts, but the most significant improvement has been against the large horizontal platform vendors. Our win rate against the top three horizontal competitors improved from 31% to 37% over the past year. Against vertical-specific competitors, we have maintained our already-strong win rate of approximately 42%. The key differentiator has been our AI capabilities — in competitive evaluations where AI analytics is a factor, our win rate jumps to 52%, which is remarkable in a market this competitive.

Operator: Our final question today comes from Amanda Liu at Bank of America. Please go ahead.

Amanda Liu, Analyst, Bank of America: Thank you. David, looking ahead to fiscal 2026, the 16% to 18% growth framework you mentioned — what are the key swing factors that would put you at the high end versus the low end of that range?

David Park, CEO: Amanda, great question. The factors that would push us toward the high end are: first, continued AI adoption — if AI analytics penetration reaches 35% of our customer base by mid-FY26, that would be a meaningful tailwind. Second, international acceleration — if Japan and Germany ramp faster than our base case, that adds directly to growth. Third, the macro environment — if we see sales cycles normalize in financial services and media, the pent-up demand in our pipeline would convert to revenue quickly.

The factors that could keep us at the lower end would be persistent macro headwinds in key verticals, slower-than-expected AI monetization if customers push back on pricing, or competitive responses from the large platform vendors. But I want to be clear — even at the low end of 16%, we would still be gaining market share significantly faster than the overall market growth rate of 8% to 10%.

Amanda Liu, Analyst, Bank of America: That is very clear. And Lisa, one final one — the $2 billion buyback authorization. How should we think about the pace of repurchases in FY26?

Lisa Martinez, CFO: We plan to be systematic about repurchases, averaging approximately $500 million per quarter through FY26. We will be opportunistic around timing, potentially accelerating during any market dislocations. Combined with our growing free cash flow, we believe this level of capital return is sustainable while still funding our organic growth investments.

Operator: That concludes our question-and-answer session. I will now turn the call back to David Park for closing remarks.

David Park, CEO: Thank you, everyone, for joining us today. Q3 was an excellent quarter that demonstrates the strength of our platform, the durability of our growth, and the expanding profitability of our business. We are executing well across all dimensions — product innovation, customer acquisition, international expansion, and operational efficiency. I am incredibly proud of the NovaTech team and confident in our trajectory as we head into Q4 and fiscal year 2026. Thank you and have a good evening.

Operator: This concludes the NovaTech Solutions Third Quarter Fiscal Year 2025 Earnings Call. You may now disconnect.
"""

print(f"Transcript length: {len(TRANSCRIPT)} characters, ~{len(TRANSCRIPT.split())} words")

## 2. Speaker Identification

Parse the transcript to identify **who said what**. We use regex-based detection since this transcript has explicit speaker labels.

In [ ]:
def identify_speakers(transcript: str) -> List[Dict]:
    """Parse transcript into speaker-labeled segments."""
    
    # Pattern: "Name, Title:" or "Name, Title, Company:"
    speaker_pattern = re.compile(
        r"^([A-Z][a-zA-Z\s]+(?:,\s*[A-Za-z\s,]+)?):(?=\s)",
        re.MULTILINE
    )
    
    segments = []
    matches = list(speaker_pattern.finditer(transcript))
    
    for i, match in enumerate(matches):
        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(transcript)
        speaker_full = match.group(1).strip()
        text = transcript[start:end].strip()
        
        # Classify role
        speaker_lower = speaker_full.lower()
        if "operator" in speaker_lower:
            role = "Operator"
        elif "ceo" in speaker_lower or "chief executive" in speaker_lower:
            role = "Executive-CEO"
        elif "cfo" in speaker_lower or "chief financial" in speaker_lower:
            role = "Executive-CFO"
        elif "vp" in speaker_lower or "vice president" in speaker_lower:
            role = "Executive-Other"
        elif "analyst" in speaker_lower:
            role = "Analyst"
        else:
            role = "Unknown"
        
        # Extract just the name
        name = speaker_full.split(",")[0].strip()
        
        segments.append({
            "speaker": name,
            "speaker_full": speaker_full,
            "role": role,
            "text": text,
            "word_count": len(text.split())
        })
    
    return segments

# Run speaker identification
segments = identify_speakers(TRANSCRIPT)

# Display results
print(f"Found {len(segments)} speaker segments\n")
print(f"{'Speaker':<25} {'Role':<18} {'Words':>6}")
print("-" * 52)
for seg in segments:
    print(f"{seg['speaker']:<25} {seg['role']:<18} {seg['word_count']:>6}")

# Summary by role
print("\n--- Word Count by Role ---")
role_words = {}
for seg in segments:
    role_words[seg["role"]] = role_words.get(seg["role"], 0) + seg["word_count"]
for role, wc in sorted(role_words.items(), key=lambda x: -x[1]):
    print(f"  {role:<18} {wc:>6} words")

## 3. Topic Segmentation

Use the LLM to break the earnings call into **thematic sections** (financial overview, segment results, guidance, Q&A topics, etc.).

In [ ]:
def segment_topics(speaker_segments: List[Dict]) -> List[Dict]:
    """Use LLM to segment transcript into topical sections."""
    
    # Build readable transcript from speaker segments
    transcript_text = "\n\n".join(
        f"[{seg['role']}] {seg['speaker']}:\n{seg['text']}"
        for seg in speaker_segments
        if seg["role"] != "Operator"  # Skip operator lines
    )
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": """You are a financial analyst. Segment this earnings call 
transcript into distinct topical sections.

For each segment, provide:
- topic: Short label (e.g., "Revenue Overview", "Cloud Segment Performance", 
  "AI Analytics Adoption", "Q4 Guidance", "Analyst Q&A: Sales Cycles")
- section_type: "prepared_remarks" or "qa"
- speakers: List of speakers in this segment
- summary: 2-3 sentence summary of key points
- key_quotes: 1-2 important direct quotes from this segment

Return JSON with a "segments" array. Aim for 8-12 segments."""},
            {"role": "user", "content": transcript_text}
        ],
        response_format={"type": "json_object"},
        temperature=0.1,
    )
    
    result = json.loads(response.choices[0].message.content)
    return result.get("segments", [])

# Run topic segmentation
topics = segment_topics(segments)

print(f"Identified {len(topics)} topic segments:\n")
for i, topic in enumerate(topics, 1):
    section = topic.get("section_type", "unknown")
    speakers = ", ".join(topic.get("speakers", []))
    print(f"{i:2d}. [{section}] {topic['topic']}")
    print(f"    Speakers: {speakers}")
    print(f"    Summary: {topic.get('summary', 'N/A')[:120]}...")
    print()

## 4. Financial Metric Extraction

Extract all **hard financial numbers** from the transcript with structured output — value, unit, type (actual vs. guidance), period, and source quote.

In [ ]:
def extract_metrics(transcript: str) -> List[Dict]:
    """Extract financial metrics with structured output."""
    
    # Filter to executive segments only (skip operator/analyst questions)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": """Extract ALL financial metrics mentioned in this 
earnings call transcript.

For each metric, return:
- metric_name: Descriptive name (e.g., "Total Revenue", "Cloud Revenue YoY Growth")
- value: The numeric value
- unit: One of "dollars_billions", "dollars_millions", "percentage", 
  "dollars_per_share", "basis_points", "count", "ratio"
- metric_type: "actual" (reported result) or "guidance" (forward-looking)
- period: Time period (e.g., "Q3 FY2025", "FY2025", "FY2026")
- speaker: Who stated this metric
- source_quote: The EXACT sentence from the transcript containing this number

Be thorough — capture every number mentioned. Return JSON with a "metrics" array."""},
            {"role": "user", "content": transcript}
        ],
        response_format={"type": "json_object"},
        temperature=0.0,
    )
    
    result = json.loads(response.choices[0].message.content)
    return result.get("metrics", [])

# Run metric extraction
metrics = extract_metrics(TRANSCRIPT)

print(f"Extracted {len(metrics)} financial metrics\n")

# Display as a DataFrame for better readability
df_metrics = pd.DataFrame(metrics)
display_cols = ["metric_name", "value", "unit", "metric_type", "period"]
available_cols = [c for c in display_cols if c in df_metrics.columns]
print(df_metrics[available_cols].to_string(index=False))

In [ ]:
# Separate actuals from guidance
actuals = [m for m in metrics if m.get("metric_type") == "actual"]
guidance = [m for m in metrics if m.get("metric_type") == "guidance"]

print(f"=== ACTUAL RESULTS ({len(actuals)} metrics) ===")
for m in actuals[:10]:  # Show first 10
    print(f"  {m['metric_name']}: {m['value']} ({m.get('unit', 'N/A')}) — {m.get('period', 'N/A')}")

print(f"\n=== FORWARD GUIDANCE ({len(guidance)} metrics) ===")
for m in guidance:
    print(f"  {m['metric_name']}: {m['value']} ({m.get('unit', 'N/A')}) — {m.get('period', 'N/A')}")

## 5. Sentiment Analysis by Topic

Score each topic segment on **polarity** (-1 to +1), **confidence** (0 to 1), and **specificity** (0 to 1). This reveals which parts of the business management feels strongest or most cautious about.

In [ ]:
def analyze_sentiment_per_topic(topics: List[Dict]) -> List[Dict]:
    """Analyze sentiment for each topic segment."""
    
    # Build a concise representation of each topic
    topics_text = "\n\n".join(
        f"--- Topic: {t['topic']} ---\nSummary: {t.get('summary', 'N/A')}\n"
        f"Key Quotes: {json.dumps(t.get('key_quotes', []))}"
        for t in topics
    )
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": """Analyze the sentiment of each topic from this 
earnings call. For each topic, score:

1. polarity: -1.0 (very negative) to +1.0 (very positive)
2. confidence: 0.0 (uncertain/hedging) to 1.0 (very confident) 
3. specificity: 0.0 (vague) to 1.0 (concrete/data-driven)
4. overall_tone: "bullish", "cautiously_optimistic", "neutral", 
   "cautiously_negative", or "bearish"
5. key_signal: One sentence explaining the most important sentiment signal
6. red_flags: Any concerning language patterns (empty list if none)

Return JSON with a "sentiments" array, one entry per topic."""},
            {"role": "user", "content": topics_text}
        ],
        response_format={"type": "json_object"},
        temperature=0.1,
    )
    
    result = json.loads(response.choices[0].message.content)
    return result.get("sentiments", [])

# Run sentiment analysis
sentiments = analyze_sentiment_per_topic(topics)

print(f"Sentiment analysis for {len(sentiments)} topics:\n")
print(f"{'Topic':<35} {'Polarity':>9} {'Confid.':>8} {'Specif.':>8} {'Tone':<25}")
print("-" * 90)
for s in sentiments:
    topic = s.get("topic", "Unknown")[:34]
    pol = s.get("polarity", 0)
    conf = s.get("confidence", 0)
    spec = s.get("specificity", 0)
    tone = s.get("overall_tone", "N/A")
    print(f"{topic:<35} {pol:>+8.2f} {conf:>8.2f} {spec:>8.2f} {tone:<25}")
    
    red_flags = s.get("red_flags", [])
    if red_flags:
        for flag in red_flags:
            print(f"    ⚠ RED FLAG: {flag}")

In [ ]:
# Visualize sentiment across topics
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

topic_labels = [s.get("topic", f"Topic {i}")[:25] for i, s in enumerate(sentiments)]
polarities = [s.get("polarity", 0) for s in sentiments]
confidences = [s.get("confidence", 0) for s in sentiments]
specificities = [s.get("specificity", 0) for s in sentiments]

# Polarity chart
colors = ["#22c55e" if p > 0 else "#ef4444" for p in polarities]
axes[0].barh(topic_labels, polarities, color=colors, edgecolor="white", linewidth=0.5)
axes[0].set_xlabel("Polarity (-1 to +1)")
axes[0].set_title("Sentiment Polarity by Topic")
axes[0].axvline(x=0, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_xlim(-1.1, 1.1)

# Confidence chart  
axes[1].barh(topic_labels, confidences, color="#3b82f6", edgecolor="white", linewidth=0.5)
axes[1].set_xlabel("Confidence (0 to 1)")
axes[1].set_title("Management Confidence by Topic")
axes[1].set_xlim(0, 1.1)

# Specificity chart
axes[2].barh(topic_labels, specificities, color="#f59e0b", edgecolor="white", linewidth=0.5)
axes[2].set_xlabel("Specificity (0 to 1)")
axes[2].set_title("Statement Specificity by Topic")
axes[2].set_xlim(0, 1.1)

plt.tight_layout()
plt.show()

print("\nInterpretation: High polarity + High confidence + High specificity = strong signal.")
print("Low specificity with high confidence = potential red flag (vague optimism).")

## 6. Forward-Looking Statement Detection

Identify all **guidance, projections, and commitments** that can be tracked in future quarters.

In [ ]:
def detect_forward_looking_statements(transcript: str) -> List[Dict]:
    """Detect and classify forward-looking statements."""
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": """Identify ALL forward-looking statements in this 
earnings call transcript.

For each forward-looking statement, extract:
- statement: The verbatim quote from the transcript
- speaker: Who made the statement
- category: One of "revenue_guidance", "margin_target", "product_initiative",
  "hiring", "capex", "market_expansion", "cost_reduction", 
  "capital_return", "competitive", "other"
- time_horizon: When expected (e.g., "Q4 FY2025", "FY2026", "FY27")
- specificity: "quantitative" (has numbers) or "qualitative" (directional only)
- confidence_language: "committed" (will/shall), "expected" (expect/anticipate), 
  or "aspirational" (hope/aim/target)
- trackable: true if this can be verified in a future quarter, false otherwise

Return JSON with a "statements" array. Be thorough."""},
            {"role": "user", "content": transcript}
        ],
        response_format={"type": "json_object"},
        temperature=0.0,
    )
    
    result = json.loads(response.choices[0].message.content)
    return result.get("statements", [])

# Run FLS detection
fls = detect_forward_looking_statements(TRANSCRIPT)

print(f"Detected {len(fls)} forward-looking statements\n")

# Group by category
from collections import defaultdict
fls_by_category = defaultdict(list)
for stmt in fls:
    fls_by_category[stmt.get("category", "other")].append(stmt)

for category, stmts in sorted(fls_by_category.items()):
    print(f"\n=== {category.upper().replace('_', ' ')} ({len(stmts)}) ===")
    for stmt in stmts:
        trackable = "TRACKABLE" if stmt.get("trackable") else "qualitative"
        horizon = stmt.get("time_horizon", "N/A")
        confidence = stmt.get("confidence_language", "N/A")
        print(f"  [{trackable}] [{horizon}] [{confidence}]")
        # Truncate long statements
        statement_text = stmt.get("statement", "N/A")
        if len(statement_text) > 120:
            statement_text = statement_text[:117] + "..."
        print(f"    \"{statement_text}\"")

## 7. Executive Summary Generation

Synthesize all extracted data into a **structured analyst brief** with direct quotes and citations.

In [ ]:
def generate_executive_summary(
    metrics: List[Dict],
    sentiments: List[Dict],
    forward_looking: List[Dict],
    topics: List[Dict],
) -> str:
    """Generate a structured executive summary."""
    
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": """You are a senior equity research analyst writing an 
earnings call summary for institutional investors.

Using the extracted data, generate a structured executive summary with these sections:

1. HEADLINE (one sentence capturing the most important takeaway)

2. KEY METRICS TABLE (markdown table with: Metric | Value | vs Prior | Commentary)
   Include: Revenue, Cloud Revenue, EPS, Operating Margin, Free Cash Flow, NRR, RPO

3. KEY TAKEAWAYS (3-5 bullet points, each with a supporting direct quote)

4. SEGMENT ANALYSIS
   - Cloud Platform: performance, growth drivers, metrics
   - Enterprise Software: performance, migration dynamics
   - International: growth, investment, margin impact

5. GUIDANCE CHANGES (what was raised/maintained/lowered and by how much)

6. RISK FACTORS & RED FLAGS (any concerns from sentiment analysis or Q&A)

7. NOTABLE Q&A EXCHANGES (2-3 most important analyst questions and answers)

RULES:
- Every factual claim must have a direct quote with speaker attribution
- Use EXACT numbers from the extracted metrics
- Keep language professional and factual — no editorializing
- Use markdown formatting"""},
            {"role": "user", "content": json.dumps({
                "company": "NovaTech Solutions (NVTS)",
                "quarter": "Q3 FY2025",
                "metrics": metrics[:30],  # Top 30 metrics
                "sentiments": sentiments,
                "forward_looking": forward_looking[:15],  # Top 15 FLS
                "topics": [{"topic": t["topic"], "summary": t.get("summary", "")}
                           for t in topics]
            }, indent=2)}
        ],
        temperature=0.3,
        max_tokens=3000,
    )
    
    return response.choices[0].message.content

# Generate the executive summary
summary = generate_executive_summary(metrics, sentiments, fls, topics)

print("=" * 80)
print("EXECUTIVE SUMMARY — NovaTech Solutions (NVTS) Q3 FY2025")
print("=" * 80)
print(summary)

## 8. Pipeline Summary & Costs

Review what the full pipeline produced and estimate API costs.

In [ ]:
# Pipeline output summary
print("=" * 60)
print("EARNINGS CALL ANALYZER — PIPELINE RESULTS")
print("=" * 60)
print(f"\nCompany:           NovaTech Solutions (NVTS)")
print(f"Quarter:           Q3 FY2025")
print(f"Transcript words:  ~{len(TRANSCRIPT.split())}")
print(f"\n--- Pipeline Outputs ---")
print(f"Speaker segments:       {len(segments)}")
print(f"Topic segments:         {len(topics)}")
print(f"Financial metrics:      {len(metrics)}")
print(f"  - Actual results:     {len([m for m in metrics if m.get('metric_type') == 'actual'])}")
print(f"  - Forward guidance:   {len([m for m in metrics if m.get('metric_type') == 'guidance'])}")
print(f"Sentiment scores:       {len(sentiments)}")
print(f"Forward-looking stmts:  {len(fls)}")
print(f"  - Trackable:          {len([f for f in fls if f.get('trackable')])}")
print(f"Executive summary:      {len(summary.split())} words")
print(f"\n--- Estimated API Cost ---")

# Rough token estimates (GPT-4o-mini pricing: $0.15/1M input, $0.60/1M output)
est_input_tokens = len(TRANSCRIPT.split()) * 1.3 * 4  # 4 LLM calls, ~1.3 tokens/word
est_output_tokens = 5000  # Estimated total output across all calls
cost_input = (est_input_tokens / 1_000_000) * 0.15
cost_output = (est_output_tokens / 1_000_000) * 0.60
total_cost = cost_input + cost_output

print(f"Est. input tokens:  ~{int(est_input_tokens):,}")
print(f"Est. output tokens: ~{est_output_tokens:,}")
print(f"Est. total cost:    ${total_cost:.4f} (GPT-4o-mini)")
print(f"Est. cost with GPT-4o: ${total_cost * 33:.2f}")
print(f"\nManual analysis time: ~3-4 hours")
print(f"Pipeline analysis time: <1 minute")

## Conclusion

This notebook demonstrated a complete **Earnings Call Analyzer** pipeline:

| Stage | What it does | Key output |
|-------|-------------|------------|
| Speaker ID | Parses who said what | Role-classified segments |
| Topic Segmentation | Breaks call into themes | 8-12 topical sections |
| Metric Extraction | Pulls financial numbers | Structured metrics with sources |
| Sentiment Analysis | Scores tone per topic | Polarity / confidence / specificity |
| FLS Detection | Finds guidance & commitments | Trackable forward-looking statements |
| Executive Summary | Synthesizes analyst brief | Cited, structured report |

**Production enhancements** would include:
- Real-time processing during live calls
- Quarter-over-quarter comparison database
- Multi-language support
- Whisper integration for audio input
- Regulatory compliance (MNPI handling)
- Dashboard integration for portfolio managers